[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/17_django/lab02_serving_a_model_with_auth.ipynb)

# 🧪 Lab 2 — Serving a churn model with Django: API, auth & history

> **Module:** Django for AI Web Apps (Module 17) · **Estimated time:** ~60 minutes · **Difficulty:** Intermediate → Advanced

[Lab 1](lab01_django_in_a_notebook.ipynb) rebuilt Django's engine room — settings, models, the ORM, templates, views, forms — one piece at a time. This lab ships the product. We train a **real scikit-learn churn model**, dump it to disk with `joblib`, load it **once** behind a Django app, and serve it two ways: a **JSON API** for robots and a **validated HTML form** for humans — with every single prediction logged to the database, and the prediction log locked behind a **login**. That is the complete **ChurnScope** pattern from [serving-a-model.md](serving-a-model.md), [auth-and-history.md](auth-and-history.md) and [deployment.md](deployment.md) — running live in this kernel, no server, no browser.

> 🧭 **Mental model: an artifact behind an endpoint.** A served model is four verbs, in strict order —
> **train offline → ship a file → load once at startup → predict per request** (and *log every prediction*).
> Training happens on your schedule, in a batch job. What crosses into the web app is a *file*. The app pays
> the loading cost once, at boot, and every request after that finds a warm model, spends milliseconds, and
> leaves a `Prediction` row behind as its receipt. Every section of this lab is one of those verbs.

**Prerequisites.** [Lab 1](lab01_django_in_a_notebook.ipynb) explains the notebook-boot trick slowly (settings, app registration, string templates) — this lab replays it in *one condensed cell* and moves on, so skim Lab 1 first if the setup cell looks like sorcery. NB 17 (scikit-learn pipelines) and NB 45 (from notebook to project) are helpful but not required. Django is the only extra dependency — §1 checks. **Optional module**: skip or skim guilt-free.

## 🎯 Learning objectives

By the end of this lab you can:

1. Train a scikit-learn churn **pipeline** offline, ship it as a **versioned `joblib` artifact**, and load it **once at startup** — never per request.
2. Serve predictions through a **JSON API** (`POST /api/score/`) that validates input, returns clean errors (400/405), and logs every score through the ORM.
3. Serve the *same* model through a **validated HTML form**, and explain what `@csrf_exempt` disables, why the JSON API needs it, and what production uses instead.
4. Protect a page with **`@login_required`**, create users safely with `create_user`, and walk the **302 → login → 200** flow with the test client.
5. Compress the whole app into a **test suite** — status codes, probability bounds, auth boundaries — that runs in milliseconds without a server.
6. Audit the app against a **production checklist** (DEBUG, SECRET_KEY, ALLOWED_HOSTS, gunicorn, collectstatic) and read the example app's real **Dockerfile** line by line.

## 1. Smoke test — is the toolbox complete?

Two toolboxes, actually. The **modelling** side — scikit-learn, joblib, pandas, numpy — is core course equipment (NB 17 onwards) and preinstalled on Colab. The **web** side is Django, this module's one extra dependency (`pip install django`, or uncomment the `%pip` line on Colab). As in Lab 1, every Django cell checks `HAS_DJANGO` and prints a skip note instead of crashing — the *modelling* cells run either way.

In [1]:
# On Colab or a fresh machine: uncomment the next line, run it, then restart the kernel.
# %pip install django

try:
    import django
    HAS_DJANGO = True
    print(f"Django {django.get_version()} — ready.")
except ImportError:
    HAS_DJANGO = False
    print("Django not installed — install with:  pip install django")
    print("Django cells below will print a skip note; the modelling cells still run.")

import joblib
import numpy as np
import pandas as pd
import sklearn

print(f"scikit-learn {sklearn.__version__} · pandas {pd.__version__} · joblib ready")

Django 5.2.15 — ready.


scikit-learn 1.8.0 · pandas 3.0.3 · joblib ready


## 2. Boot ChurnScope's engine — Lab 1, condensed to one cell

Everything Lab 1 built across four sections, restored in one breath: configure settings by hand, register a `scoring` app (an app is just an importable package — we write a two-file stub into a temp folder), point the database at a fresh SQLite file, and `migrate`. If any line looks mysterious, [Lab 1](lab01_django_in_a_notebook.ipynb) §2 walks it slowly — including the 🔬 story behind `DJANGO_ALLOW_ASYNC_UNSAFE` (Jupyter's kernel runs an asyncio event loop, and Django's ORM refuses to make blocking database calls inside one unless you set that flag; it's the standard notebook escape hatch).

Three things are **new** relative to Lab 1's settings, and all three exist because *this* lab has users:

| Setting | Lab 1 | This lab | Why |
|---|---|---|---|
| `INSTALLED_APPS` | contenttypes, auth, `scoring` | + **`django.contrib.sessions`** | a login *is* a session — it needs a table to live in |
| `MIDDLEWARE` | *(none needed)* | **Session → Csrf → Authentication** | sessions attach the cookie, CSRF guards forms, auth turns the session into `request.user` |
| auth redirects | — | `LOGIN_URL`, `LOGIN_REDIRECT_URL`, `LOGOUT_REDIRECT_URL` | the three signposts from [auth-and-history.md](auth-and-history.md) |

One more small addition: the `auth` **context processor**, so templates automatically receive the current `user` — a convenience `startproject`'s default settings include and our hand-rolled `TEMPLATES` must opt into.

In [2]:
import os
import sys
import tempfile
from pathlib import Path

if HAS_DJANGO:
    from django.conf import settings

    # Jupyter runs an asyncio event loop; without this flag every ORM call would
    # raise SynchronousOnlyOperation. Standard notebook fix — Lab 1 §2 has the 🔬.
    os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

    LAB_DIR = Path(tempfile.gettempdir()) / "churnscope_lab2"
    DB_PATH = LAB_DIR / "churnscope_lab2.sqlite3"
    # One dict = the whole template store (locmem loader). Survives re-runs:
    TEMPLATE_REGISTRY = globals().get("TEMPLATE_REGISTRY", {})

    if settings.configured:
        print("Settings already configured — restart the kernel to reconfigure.")
    else:
        # The 'scoring' app: an importable two-file package (Lab 1's trick).
        (LAB_DIR / "scoring").mkdir(parents=True, exist_ok=True)
        (LAB_DIR / "scoring" / "__init__.py").write_text(
            "# The scoring app — same name as ChurnScope's app package.\n")
        (LAB_DIR / "scoring" / "models.py").write_text(
            "# Models are defined in notebook cells instead (see Lab 1 §3).\n")
        sys.path.insert(0, str(LAB_DIR))

        if DB_PATH.exists():                     # fresh database per kernel session
            DB_PATH.unlink()

        settings.configure(
            DEBUG=True,
            SECRET_KEY="lab-only-not-a-real-secret",
            ALLOWED_HOSTS=["testserver"],        # the test client's hostname
            INSTALLED_APPS=[
                "django.contrib.contenttypes",
                "django.contrib.auth",           # User, permissions, login views
                "django.contrib.sessions",       # NEW: logins live in sessions
                "scoring",
            ],
            MIDDLEWARE=[                         # NEW: the auth plumbing, in order
                "django.contrib.sessions.middleware.SessionMiddleware",
                "django.middleware.csrf.CsrfViewMiddleware",
                "django.contrib.auth.middleware.AuthenticationMiddleware",
            ],
            DATABASES={"default": {
                "ENGINE": "django.db.backends.sqlite3",
                "NAME": DB_PATH,
            }},
            ROOT_URLCONF="notebook_urls",        # a module we build in §4
            TEMPLATES=[{
                "BACKEND": "django.template.backends.django.DjangoTemplates",
                "OPTIONS": {
                    "loaders": [
                        ("django.template.loaders.locmem.Loader", TEMPLATE_REGISTRY),
                    ],
                    "context_processors": [      # NEW: hand every template `user`
                        "django.contrib.auth.context_processors.auth",
                    ],
                },
            }],
            # The three auth signposts from auth-and-history.md:
            LOGIN_URL="/accounts/login/",        # where @login_required sends strangers
            LOGIN_REDIRECT_URL="/history/",      # where a fresh login lands
            LOGOUT_REDIRECT_URL="/score/",       # where logout drops you
            USE_TZ=True,
            TIME_ZONE="UTC",
            DEFAULT_AUTO_FIELD="django.db.models.BigAutoField",
        )
        django.setup()                           # populate the app registry — once!

    from django.apps import apps

    print("Installed apps:", [a.label for a in apps.get_app_configs()])
    print("Database file :", settings.DATABASES["default"]["NAME"])
else:
    print("Django not installed — skipping.")

Installed apps: ['contenttypes', 'auth', 'sessions', 'scoring']
Database file : /var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/churnscope_lab2/churnscope_lab2.sqlite3


And the furniture from Lab 1's later sections — the **`Prediction` model** and the **`ChurnForm`** — field-for-field from `example-app/scoring/models.py` and `forms.py` (plus the notebook-only `app_label`), followed immediately by `migrate`. The model is our audit trail: what went in, what came out, when.

In [3]:
if HAS_DJANGO:
    from django import forms
    from django.core.management import call_command
    from django.db import connection, models

    class Prediction(models.Model):
        """One churn score, logged for the dashboard and audit trail."""

        created_at = models.DateTimeField(auto_now_add=True)
        tenure_months = models.PositiveIntegerField()
        monthly_charges = models.FloatField()
        support_tickets = models.PositiveIntegerField()
        contract = models.CharField(max_length=20)
        probability = models.FloatField()
        will_churn = models.BooleanField()

        class Meta:
            app_label = "scoring"        # notebook-only: pin the model to our app
            ordering = ["-created_at"]   # newest first — everywhere, by default

        def __str__(self):
            verdict = "churn" if self.will_churn else "stay"
            return f"{self.contract}: p={self.probability:.2f} ({verdict})"

    CONTRACT_CHOICES = [
        ("month-to-month", "Month-to-month"),
        ("one-year", "One year"),
        ("two-year", "Two year"),
    ]

    class ChurnForm(forms.Form):
        """Validates one customer's details before they reach the scorer."""

        tenure_months = forms.IntegerField(min_value=0, max_value=120, initial=6)
        monthly_charges = forms.FloatField(min_value=0, initial=70.0)
        support_tickets = forms.IntegerField(min_value=0, max_value=50, initial=2)
        contract = forms.ChoiceField(choices=CONTRACT_CHOICES)

    call_command("migrate", run_syncdb=True, verbosity=0)   # ≙ manage.py migrate

    tables = sorted(connection.introspection.table_names())
    print(f"{len(tables)} tables — including:")
    for name in tables:
        if name in ("scoring_prediction", "auth_user", "django_session"):
            print(f"  {name:<20} ← {'ours' if name.startswith('scoring') else 'a battery'}")
else:
    print("Django not installed — skipping.")

10 tables — including:
  auth_user            ← a battery
  django_session       ← a battery
  scoring_prediction   ← ours


## 3. Train offline, ship a file, load once

[serving-a-model.md](serving-a-model.md) ends with a three-step recipe for swapping ChurnScope's transparent stand-in for a real trained model: **(1) train and dump outside Django, (2) load once at startup, (3) keep the signature.** This section performs all three, live.

First, the stand-in itself — `example-app/scoring/scorer.py`, coefficient for coefficient. In the example app it *pretends* to be a model; here we give it a more honest job: it is our **data-generating process**. We'll synthesise a customer table whose churn behaviour follows exactly this logic, train a `LogisticRegression` on it, and check the model *recovers the story* — month-to-month contracts and support tickets push risk up, tenure pushes it down. (With real data you'd skip this step and load your NB 33-style customer table; synthetic-with-known-truth is how you rehearse the machinery.)

In [4]:
import math

CONTRACT_RISK = {"month-to-month": 0.9, "one-year": 0.0, "two-year": -0.9}


def baseline_churn_probability(*, tenure_months, monthly_charges, support_tickets, contract):
    """The example app's transparent stand-in — here, our ground truth."""
    z = (
        -0.5
        + 1.4 * CONTRACT_RISK.get(contract, 0.0)   # month-to-month is risky
        - 0.05 * float(tenure_months)               # loyalty lowers risk
        + 0.015 * float(monthly_charges)            # pricier plans churn more
        + 0.25 * float(support_tickets)             # friction churns
    )
    return 1.0 / (1.0 + math.exp(-z))


rng = np.random.default_rng(42)
N = 600

customers = pd.DataFrame({
    "tenure_months": rng.integers(0, 73, size=N),
    "monthly_charges": np.round(rng.uniform(20, 120, size=N), 2),
    "support_tickets": rng.poisson(1.5, size=N).clip(max=12),
    "contract": rng.choice(["month-to-month", "one-year", "two-year"],
                           size=N, p=[0.55, 0.25, 0.20]),
})
p_true = customers.apply(lambda row: baseline_churn_probability(**row), axis=1)
customers["churned"] = (rng.random(N) < p_true).astype(int)   # noisy, like real life

print(f"{N} customers · churn rate {customers['churned'].mean():.1%}")
customers.head()

600 customers · churn rate 43.0%


,tenure_months,monthly_charges,support_tickets,contract,churned
0,6,97.90,1,month-to-month,1
1,56,33.46,0,one-year,0
2,47,73.61,3,two-year,0
3,32,71.42,0,month-to-month,0
4,31,105.76,1,month-to-month,1


**Step 1 — train and dump, outside Django.** The estimator is a proper **`Pipeline`** (NB 17): a `ColumnTransformer` that standardises the three numeric columns and one-hot encodes `contract`, feeding a `LogisticRegression`. The pipeline *is* the artifact — preprocessing and model travel as one object, so serving code can never apply the wrong scaling.

> ⚠️ **`handle_unknown="ignore"` is a serving decision, made at training time.** The default `OneHotEncoder` *raises* on categories it never saw — which means one customer with `contract="quarterly"` would crash your API at 3 a.m. With `"ignore"` an unseen category encodes as all-zeros: the model quietly falls back to "no contract signal". Decide *how the model should fail* before you ship it, not after.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FEATURES = ["tenure_months", "monthly_charges", "support_tickets", "contract"]
NUMERIC = ["tenure_months", "monthly_charges", "support_tickets"]

X_train, X_test, y_train, y_test = train_test_split(
    customers[FEATURES], customers["churned"],
    test_size=0.25, random_state=42, stratify=customers["churned"])

pipeline = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("contract", OneHotEncoder(handle_unknown="ignore"), ["contract"]),
    ])),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipeline.fit(X_train, y_train)

auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
print(f"held-out test set: AUC = {auc:.3f} · accuracy = {pipeline.score(X_test, y_test):.3f}")
print("(labels were sampled with noise, so even the true model couldn't score 1.0)")

held-out test set: AUC = 0.873 · accuracy = 0.807
(labels were sampled with noise, so even the true model couldn't score 1.0)


Good enough to ship. Now the hand-off: in real life this training script and the web app are **different processes on different schedules** — training is a batch job (cron, CI, Module 13's schedulers); the app just receives a file. `joblib.dump` is the hand-off, and a **version string** rides along, because six months from now "which model produced this score?" will be a real question with real consequences.

> ⚠️ **Version your artifacts.** A model file with no version is a mystery in a trench coat. The chapter's convention — a `MODEL_VERSION` constant next to the artifact it describes, logged with every prediction — is two lines of insurance. (The stretch exercises write it into the `Prediction` table itself.)

In [6]:
MODEL_VERSION = "2026-07-logreg-v1"          # date + family + revision
MODEL_PATH = LAB_DIR if HAS_DJANGO else Path(tempfile.gettempdir())
MODEL_PATH = MODEL_PATH / "churn_model.joblib"

joblib.dump(pipeline, MODEL_PATH)
print(f"shipped: {MODEL_PATH}")
print(f"         {MODEL_PATH.stat().st_size / 1024:.1f} KB · version {MODEL_VERSION}")

shipped: /var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/churnscope_lab2/churn_model.joblib
         3.3 KB · version 2026-07-logreg-v1


**Step 2 — load once at startup. Step 3 — keep the signature.** The serving side never touches `pipeline` — it loads the *file*, once, into a module-level cache, exactly like the chapter's `scorer.py`. And the public function keeps the stand-in's signature — same name, same keyword-only arguments, same 0-to-1 return — so views, forms and templates can't tell the difference. That signature is **the seam**: everything on one side is ML, everything on the other side is web.

> ⚠️ **Never load — and *never* train — per request.** `joblib.load` in a view means disk I/O plus unpickling on the hot path, thousands of times, for a file that changes only at deploy time; training in a view is that mistake squared. In the real app the one-time load lives in `ScoringConfig.ready()` (`scoring/apps.py`) — Django's "the app just booted" hook. A notebook's equivalent of *startup* is simply: this cell, run once.

In [7]:
_model = None                       # module-level cache — filled once, read per request


def load_model():
    """Pay the loading cost ONCE. In the real app, ScoringConfig.ready() calls this."""
    global _model
    _model = joblib.load(MODEL_PATH)
    return _model


def churn_probability(*, tenure_months, monthly_charges, support_tickets, contract):
    """Same signature as the stand-in — the seam the whole app talks to.

    The pipeline was trained on a DataFrame with named columns, so serving
    must rebuild that exact schema: same columns, same names, one row.
    """
    row = pd.DataFrame([{
        "tenure_months": int(tenure_months),
        "monthly_charges": float(monthly_charges),
        "support_tickets": int(support_tickets),
        "contract": str(contract),
    }])
    return float(_model.predict_proba(row)[0, 1])    # column 1 = P(churn)


load_model()                        # ← "startup" happens here, exactly once

demo = dict(tenure_months=2, monthly_charges=95.0, support_tickets=5,
            contract="month-to-month")
print(f"trained model : P(churn) = {churn_probability(**demo):.3f}")
print(f"stand-in      : P(churn) = {baseline_churn_probability(**demo):.3f}")
print("→ the LogisticRegression recovered the stand-in's story from noisy labels")

trained model : P(churn) = 0.956
stand-in      : P(churn) = 0.966
→ the LogisticRegression recovered the stand-in's story from noisy labels


---

### ✋ Quick exercise (~2 min) — Risky vs safe, through the seam

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The example app's test suite pins one business fact: *a month-to-month customer must out-risk an otherwise identical two-year customer.* Verify our trained model kept that promise — score the **same profile** (3 months tenure, 95.0 monthly charges, 4 tickets) under `"month-to-month"` and `"two-year"`, `assert` both probabilities live in `[0, 1]`, and `assert` the risky one is larger. Call only `churn_probability(...)` — the seam, not the pipeline.

In [8]:
# ✍️ Your turn 👇
profile = dict(tenure_months=3, monthly_charges=95.0, support_tickets=4)

# p_risky = churn_probability(**profile, contract=...)
p_risky = ...
p_safe = ...

# assert 0.0 <= p_risky <= 1.0 and 0.0 <= p_safe <= 1.0
# assert p_risky > p_safe, "month-to-month should out-risk two-year"
# print(f"month-to-month {p_risky:.3f}  >  two-year {p_safe:.3f}")

<details>
<summary>✅ <b>Solution</b></summary>

```python
profile = dict(tenure_months=3, monthly_charges=95.0, support_tickets=4)

p_risky = churn_probability(**profile, contract="month-to-month")
p_safe = churn_probability(**profile, contract="two-year")

assert 0.0 <= p_risky <= 1.0 and 0.0 <= p_safe <= 1.0
assert p_risky > p_safe, "month-to-month should out-risk two-year"
print(f"month-to-month {p_risky:.3f}  >  two-year {p_safe:.3f}")
```

Those two asserts are `example-app/scoring/tests.py` (`test_month_to_month_is_riskier`, `test_probability_in_range`) written as notebook lines — §7 folds them into the full suite. Note what you *didn't* need to know: that there's a `StandardScaler`, a one-hot encoding, or a DataFrame inside. The seam hides the ML from the app — which is exactly what lets the example app swap its stand-in for this pipeline without touching a single view.
</details>

## 4. The JSON API — `POST /api/score/`

First serving surface: robots. Scripts, cron jobs, other services — the callers from Modules 9 and 8 — speak JSON, not HTML forms. The view below is `api_score` from `example-app/scoring/views.py`, now calling our *trained* model through the seam: parse JSON → validate/coerce → predict → **log the `Prediction` row** → return JSON. Bad input never reaches the model — it gets a clean `400` with a reason instead of a stack trace.

> ⚠️ **What `@csrf_exempt` really does.** CSRF protection defends *browser* users: since browsers attach session cookies automatically, a malicious page could POST to ChurnScope *as you* — so Django requires every POST to carry a token proving it came from a page Django itself rendered. A JSON API called by scripts has **no session cookie to abuse and no form to plant a token in** — the shield can't work there, so `@csrf_exempt` switches it off *for this one view*. The honest cost: the endpoint is now open, so real deployments pair it with **token authentication** (DRF or Django Ninja, per the chapter) instead of cookies. Never sprinkle `@csrf_exempt` on *form* views to make an error go away — §5 shows the shield doing its job.

In [9]:
if HAS_DJANGO:
    import json

    from django.http import JsonResponse
    from django.views.decorators.csrf import csrf_exempt
    from django.views.decorators.http import require_POST

    THRESHOLD = 0.5

    @csrf_exempt          # stateless JSON API — see the ⚠️ above
    @require_POST         # GET /api/score/ makes no sense → automatic 405
    def api_score(request):
        """POST JSON → {"probability": float, "will_churn": bool} — and log it."""
        try:
            payload = json.loads(request.body or "{}")
            data = {
                "tenure_months": int(payload["tenure_months"]),
                "monthly_charges": float(payload["monthly_charges"]),
                "support_tickets": int(payload["support_tickets"]),
                "contract": str(payload["contract"]),
            }
        except (KeyError, ValueError, TypeError) as exc:
            return JsonResponse({"error": f"invalid request: {exc}"}, status=400)

        p = churn_probability(**data)              # the seam, again
        will_churn = p >= THRESHOLD
        Prediction.objects.create(probability=round(p, 4), will_churn=will_churn, **data)
        return JsonResponse({"probability": round(p, 4), "will_churn": will_churn})

    print("api_score defined — a function until a URL points at it.")
else:
    print("Django not installed — skipping.")

api_score defined — a function until a URL points at it.


Route it — the URLconf is a module we assemble by hand and plant in `sys.modules` (Lab 1 §6's trick), matching the `ROOT_URLCONF = "notebook_urls"` promise from §2. Then call it with the test **`Client`**: a browser without the browser, pushing a real request through URLconf → view → model → ORM.

In [10]:
if HAS_DJANGO:
    import types

    from django.test import Client
    from django.urls import clear_url_caches, include, path

    urlconf = types.ModuleType("notebook_urls")
    urlconf.urlpatterns = [
        path("api/score/", api_score, name="api_score"),
    ]
    sys.modules["notebook_urls"] = urlconf
    clear_url_caches()          # Django caches the resolver — flush after edits

    client = Client()

    CUSTOMER = {"tenure_months": 3, "monthly_charges": 95,
                "support_tickets": 4, "contract": "month-to-month"}

    resp = client.post("/api/score/", data=json.dumps(CUSTOMER),
                       content_type="application/json")
    print("POST /api/score/ →", resp.status_code, resp.json())
    print("rows logged so far:", Prediction.objects.count())
else:
    print("Django not installed — skipping.")

POST /api/score/ → 200 {'probability': 0.9431, 'will_churn': True}
rows logged so far: 1


The same `curl` from the chapter would get the same JSON. Now the unhappy paths — a production API is *defined* by how it fails. Watch three failures produce three clean, correct responses (the red `Bad Request` / `Method Not Allowed` lines are Django's request logger agreeing with us):

In [11]:
if HAS_DJANGO:
    missing = client.post("/api/score/", data='{"monthly_charges": 95}',
                          content_type="application/json")
    print("missing fields      →", missing.status_code, missing.json())

    not_json = client.post("/api/score/", data="tenure=3&contract=gold",
                           content_type="application/json")
    print("body isn't JSON     →", not_json.status_code, not_json.json())

    wrong_method = client.get("/api/score/")
    print("GET instead of POST →", wrong_method.status_code, "(require_POST at work)")

    # And the ⚠️ from above, demonstrated: a CSRF-strict client (a stand-in for a
    # real browser without a token) still passes, because of @csrf_exempt.
    strict = Client(enforce_csrf_checks=True)
    r = strict.post("/api/score/", data=json.dumps(CUSTOMER),
                    content_type="application/json")
    print("CSRF-strict POST    →", r.status_code, "(csrf_exempt lets robots in)")
else:
    print("Django not installed — skipping.")

Bad Request: /api/score/


Bad Request: /api/score/


Method Not Allowed (GET): /api/score/


missing fields      → 400 {'error': "invalid request: 'tenure_months'"}
body isn't JSON     → 400 {'error': 'invalid request: Expecting value: line 1 column 1 (char 0)'}
GET instead of POST → 405 (require_POST at work)
CSRF-strict POST    → 200 (csrf_exempt lets robots in)


---

### ✋ Quick exercise (~2 min) — The ledger never lies

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

"Log every prediction" is a *claim* — audit it. POST a loyal customer (`tenure_months=48, monthly_charges=60, support_tickets=0, contract="two-year"`) to `/api/score/` and assert the whole contract: status `200`; `probability` in `[0, 1]`; `will_churn` consistent with `probability >= THRESHOLD`; **exactly one** new `Prediction` row (count it before and after); and the newest row (`order_by("-pk").first()`) is your two-year customer.

In [12]:
# ✍️ Your turn 👇
loyal = {"tenure_months": 48, "monthly_charges": 60,
         "support_tickets": 0, "contract": "two-year"}

# before = Prediction.objects.count()
# resp = client.post("/api/score/", data=json.dumps(loyal), content_type="application/json")
resp = ...

# body = resp.json()
# assert resp.status_code == 200
# assert 0.0 <= body["probability"] <= 1.0
# assert body["will_churn"] == (body["probability"] >= THRESHOLD)
# assert Prediction.objects.count() == before + 1
# newest = Prediction.objects.order_by("-pk").first()
# assert newest.contract == "two-year"
# print("logged:", newest)

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    loyal = {"tenure_months": 48, "monthly_charges": 60,
             "support_tickets": 0, "contract": "two-year"}

    before = Prediction.objects.count()
    resp = client.post("/api/score/", data=json.dumps(loyal),
                       content_type="application/json")
    body = resp.json()

    assert resp.status_code == 200
    assert 0.0 <= body["probability"] <= 1.0
    assert body["will_churn"] == (body["probability"] >= THRESHOLD)
    assert Prediction.objects.count() == before + 1
    newest = Prediction.objects.order_by("-pk").first()
    assert newest.contract == "two-year"
    print("logged:", newest, "| rows now:", Prediction.objects.count())
else:
    print("Django not installed — skipping.")
```

A 48-month two-year customer with zero tickets scores near zero — and the row proves the API kept its side-effect promise. Two details worth stealing: the **before/after count** (never assert absolute counts — other cells also log rows), and `order_by("-pk")` for "the row just inserted" (the model's `-created_at` ordering can tie when two rows land in the same clock tick; primary keys never tie).
</details>

## 5. The HTML form — the same seam, for humans

Second surface: people. The `index` view below is `example-app/scoring/views.py` almost line for line (the example app mounts it at the site root `/`; we mount it at `/score/`), give or take the `MODEL_VERSION` stamp the notebook adds to each result: GET shows the form, POST validates through **`ChurnForm`**, calls `churn_probability`, **logs the row**, and re-renders with the verdict. Note what it has in common with `api_score` — validate → predict → log → respond — and what differs: the form does the validating, HTML does the talking, and **CSRF stays on**, because this *is* a browser flow.

In [13]:
if HAS_DJANGO:
    from django.shortcuts import render

    TEMPLATE_REGISTRY["scoring/form.html"] = """\
<h1>ChurnScope — score a customer</h1>
<form method="post">
  {% csrf_token %}
  {{ form.as_p }}
  <button type="submit">Score</button>
</form>
{% if result %}<p id="result">Churn probability: <b>{{ result.probability|floatformat:2 }}</b> —
{% if result.will_churn %}likely to CHURN{% else %}likely to stay{% endif %}
<small>(model {{ result.model_version }})</small></p>{% endif %}"""

    def index(request):
        """The churn form: GET shows it, POST scores + logs + shows the result."""
        result = None
        if request.method == "POST":
            form = ChurnForm(request.POST)
            if form.is_valid():
                data = form.cleaned_data
                p = churn_probability(**data)
                will_churn = p >= THRESHOLD
                Prediction.objects.create(probability=round(p, 4),
                                          will_churn=will_churn, **data)
                result = {"probability": p, "will_churn": will_churn,
                          "model_version": MODEL_VERSION}
        else:
            form = ChurnForm()
        return render(request, "scoring/form.html",
                      {"form": form, "result": result, "threshold": THRESHOLD})

    urlconf.urlpatterns = [
        path("score/", index, name="index"),
        path("api/score/", api_score, name="api_score"),
    ]
    clear_url_caches()

    page = client.get("/score/")
    print("GET /score/ →", page.status_code, "(the empty form)")

    before = Prediction.objects.count()
    resp = client.post("/score/", {"tenure_months": 2, "monthly_charges": 99.5,
                                   "support_tickets": 4, "contract": "month-to-month"})
    print("POST /score/ →", resp.status_code)
    for line in resp.content.decode().splitlines():
        if 'id="result"' in line or "likely to" in line or "model " in line:
            print("   ", line.strip())
    print(f"rows: {before} → {Prediction.objects.count()}   ← the view logged it")
else:
    print("Django not installed — skipping.")

GET /score/ → 200 (the empty form)
POST /score/ → 200
    <p id="result">Churn probability: <b>0.95</b> —
    likely to CHURN
    <small>(model 2026-07-logreg-v1)</small></p>
rows: 2 → 3   ← the view logged it


And the promised counter-demonstration: the **same CSRF-strict client** that sailed through the API gets stopped cold by the form view, because a POST without the `{% csrf_token %}` stamp is exactly what a cross-site forgery looks like. (The default test `Client` politely carries a skip flag, which is why our earlier POST worked without a token — a real browser gets no such courtesy.)

In [14]:
if HAS_DJANGO:
    r = strict.post("/score/", {"tenure_months": 2, "monthly_charges": 99.5,
                                "support_tickets": 4, "contract": "month-to-month"})
    print("CSRF-strict POST /score/ →", r.status_code, "— the shield works")
    print("fix for browsers: keep {% csrf_token %} in the form (we did);")
    print("fix for robots  : don't use cookie auth at all → tokens (DRF/Ninja)")
else:
    print("Django not installed — skipping.")

Forbidden (CSRF cookie not set.): /score/


CSRF-strict POST /score/ → 403 — the shield works
fix for browsers: keep {% csrf_token %} in the form (we did);
fix for robots  : don't use cookie auth at all → tokens (DRF/Ninja)


## 6. Auth — the last battery, and the protected history page

Every score so far left a `Prediction` row behind; the **history page** is where humans read that ledger — and it must not be public. [auth-and-history.md](auth-and-history.md)'s whole pitch: you write almost no auth code, because `django.contrib.auth` (installed in §2) ships users, password hashing, sessions, and complete login/logout views. You add exactly three things — mount `django.contrib.auth.urls`, set the three redirect settings (§2 did), give the login view a `registration/login.html` template — and then protect any view with **one decorator**:

- `@login_required` bounces strangers to `LOGIN_URL` with `?next=` so they land back where they were heading;
- the view itself stays four lines: paginate the queryset, render. Newest-first ordering comes free from the model's `Meta.ordering`.

We also add a tiny `whoami` view, because the honest way to understand auth is to watch **`request.user`** change: `AuthenticationMiddleware` reads the session cookie and attaches either a real `User` or `AnonymousUser` to every request — *that attribute is what "logged in" means.*

In [15]:
if HAS_DJANGO:
    from django.contrib.auth.decorators import login_required
    from django.core.paginator import Paginator

    TEMPLATE_REGISTRY["scoring/history.html"] = """\
<h1>Prediction history</h1>
<p>page {{ page_obj.number }} of {{ page_obj.paginator.num_pages }} — signed in as <b>{{ user.username }}</b></p>
<table>
  <tr><th>#</th><th>when</th><th>contract</th><th>p</th><th>verdict</th></tr>
{% for p in page_obj %}  <tr><td>{{ forloop.counter }}</td><td>{{ p.created_at|date:"Y-m-d H:i" }}</td><td>{{ p.contract }}</td><td>{{ p.probability|floatformat:2 }}</td><td>{% if p.will_churn %}<b>churn</b>{% else %}stay{% endif %}</td></tr>
{% endfor %}</table>"""

    TEMPLATE_REGISTRY["registration/login.html"] = """\
<h1>Sign in to ChurnScope</h1>
<form method="post">
  {% csrf_token %}
  {{ form.as_p }}
  <button type="submit">Log in</button>
</form>"""

    @login_required
    def history(request):
        """Paginated prediction log, newest first (per the model's Meta.ordering)."""
        paginator = Paginator(Prediction.objects.all(), per_page=10)
        page_obj = paginator.get_page(request.GET.get("page"))
        return render(request, "scoring/history.html", {"page_obj": page_obj})

    def whoami(request):
        """Who does Django think is asking? request.user always knows."""
        return JsonResponse({"user": str(request.user),
                             "is_authenticated": request.user.is_authenticated})

    urlconf.urlpatterns = [
        path("score/", index, name="index"),
        path("api/score/", api_score, name="api_score"),
        path("history/", history, name="history"),
        path("whoami/", whoami, name="whoami"),
        # Django's complete login/logout/password machinery — one include:
        path("accounts/", include("django.contrib.auth.urls")),
    ]
    clear_url_caches()

    print("Routes:", ", ".join(f"/{p.pattern}" for p in urlconf.urlpatterns))
else:
    print("Django not installed — skipping.")

Routes: /score/, /api/score/, /history/, /whoami/, /accounts/


**Act one: a stranger knocks.** Anonymous request → `302` to the login page, with `?next=` remembering the destination — and the login page itself already works, rendered by Django's own `LoginView` through our nine-line template:

In [16]:
if HAS_DJANGO:
    print("whoami, anonymous  :", client.get("/whoami/").json())

    bounced = client.get("/history/")
    print("GET /history/      :", bounced.status_code, "→", bounced.headers["Location"])

    login_page = client.get(bounced.headers["Location"])
    print("GET the login page :", login_page.status_code, "— Django's own LoginView")
    print("\n".join(login_page.content.decode().splitlines()[:4]))
else:
    print("Django not installed — skipping.")

whoami, anonymous  : {'user': 'AnonymousUser', 'is_authenticated': False}
GET /history/      : 302 → /accounts/login/?next=/history/


GET the login page : 200 — Django's own LoginView
<h1>Sign in to ChurnScope</h1>
<form method="post">
  <input type="hidden" name="csrfmiddlewaretoken" value="ZW5fIo1rKSgIbUehq03OqpACwgCkmh70aqzyqJErO5CI0lIoe0izEpd5QZUcIGT2">
  <p>


**Act two: give someone a key.** `User.objects.create_user(...)` — and only ever that.

> ⚠️ **`create_user`, never `User(password=...)`.** The constructor stores whatever you pass — *verbatim, in plaintext*. `create_user` runs the password through Django's salted PBKDF2 hasher; look at the stored value below and count what an attacker who steals the database learns: nothing. (Same rule via `user.set_password(...)` when changing one.)

In [17]:
if HAS_DJANGO:
    from django.contrib.auth.models import User

    analyst = User.objects.filter(username="analyst").first()   # idempotent re-runs
    if analyst is None:
        analyst = User.objects.create_user("analyst", password="pass1234")

    print("user   :", analyst.username)
    print("stored :", analyst.password[:45] + "…")
    print("         ↑ algorithm, iterations, salt, hash — never the password")
else:
    print("Django not installed — skipping.")

user   : analyst
stored : pbkdf2_sha256$1000000$NkgaBeYkfxz22sTrb1JsFj$…
         ↑ algorithm, iterations, salt, hash — never the password


**Act three: walk in.** In a browser the analyst would type the password into that login form; the test client's working path is **`force_login(user)`** — it plants a valid session for the user directly, skipping the password check and hashers. That's the standard fast path in Django's own test suites: *auth flows you test once, protected views you test everywhere*, and `force_login` keeps "everywhere" fast. The full form flow is one POST, for reference — it works here too, it's just not the path we'll lean on:

```python
# the browser-faithful path (reference — force_login below is the working path):
fresh = Client()
resp = fresh.post("/accounts/login/",
                  {"username": "analyst", "password": "pass1234"}, follow=True)
# Django checks the password, creates the session, then follows the redirect
# to LOGIN_REDIRECT_URL — so resp is the /history/ page, already logged in.
```

In [18]:
if HAS_DJANGO:
    client.force_login(analyst)          # a valid session, no password dance

    resp = client.get("/history/")
    print("GET /history/ →", resp.status_code, "  (302 → login → 200: the full arc)\n")
    print("\n".join(resp.content.decode().splitlines()[:8]))
    print("…")
    print("\nwhoami, logged in:", client.get("/whoami/").json())
else:
    print("Django not installed — skipping.")

GET /history/ → 200   (302 → login → 200: the full arc)

<h1>Prediction history</h1>
<p>page 1 of 1 — signed in as <b>analyst</b></p>
<table>
  <tr><th>#</th><th>when</th><th>contract</th><th>p</th><th>verdict</th></tr>
  <tr><td>1</td><td>2026-07-03 20:30</td><td>month-to-month</td><td>0.95</td><td><b>churn</b></td></tr>
  <tr><td>2</td><td>2026-07-03 20:30</td><td>month-to-month</td><td>0.94</td><td><b>churn</b></td></tr>
  <tr><td>3</td><td>2026-07-03 20:30</td><td>month-to-month</td><td>0.94</td><td><b>churn</b></td></tr>
</table>
…

whoami, logged in: {'user': 'analyst', 'is_authenticated': True}


---

### ✋ Quick exercise (~2 min) — A second pair of eyes

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The analyst isn't the only one with a key. Create a user `"intern"` (idempotently — re-running the cell must not crash), log them in with `force_login` **on a brand-new `Client`**, and check: `/history/` returns `200` and still contains `"month-to-month"`, and `/whoami/` names the intern. Then answer the real question: whose predictions is the intern looking at — and is that a feature or a bug?

In [19]:
# ✍️ Your turn 👇
# intern = User.objects.filter(username="intern").first()
# if intern is None:
#     intern = ...
intern = ...

# second = Client()
# second.force_login(intern)
# resp = second.get("/history/")
# assert resp.status_code == 200 and "month-to-month" in resp.content.decode()
# print(second.get("/whoami/").json())

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    intern = User.objects.filter(username="intern").first()
    if intern is None:
        intern = User.objects.create_user("intern", password="intern-pw-42")

    second = Client()
    second.force_login(intern)

    resp = second.get("/history/")
    assert resp.status_code == 200
    assert "month-to-month" in resp.content.decode()
    print(second.get("/whoami/").json())
    print("→ the intern sees EVERY prediction, including the analyst's")
else:
    print("Django not installed — skipping.")
```

Both, honestly. `@login_required` answers *"are you anyone?"*, not *"are you allowed to see this row?"* — the history is one global ledger, which is fine for a small team's audit trail and wrong the moment customers log in. Authorisation (per-user filtering, permissions) is a separate layer on top of authentication — Stretch exercise territory, and the reason `Prediction` would grow a `user` foreign key in a multi-tenant ChurnScope.
</details>

## 7. The test suite — the whole app in milliseconds

Everything we just built by hand, `example-app/scoring/tests.py` pins down as tests — and the deployment chapter's point is that this is *cheap*: the test client needs no server, so the suite runs in CI in well under a second. In the real project the tests live in a `TestCase`, which gives every test a **fresh, throwaway database**:

```python
class HistoryTests(TestCase):
    def setUp(self):
        self.user = User.objects.create_user("analyst", password="pass1234")

    def test_anonymous_redirected_to_login(self):
        resp = self.client.get(reverse("scoring:history"))
        self.assertRedirects(resp, "/accounts/login/?next=/history/")

    def test_logged_in_user_sees_prediction(self):
        self.client.login(username="analyst", password="pass1234")
        resp = self.client.get(reverse("scoring:history"))
        self.assertContains(resp, "month-to-month")
```

Our notebook shares one database across cells, so we write the same checks as **plain asserts with before/after deltas** — nine of them, covering the scorer's contract, both serving surfaces, both failure modes, and both sides of the auth boundary. Read each assert as a sentence; together they *are* the spec of ChurnScope:

In [20]:
if HAS_DJANGO:
    suite = Client()                       # brand new — anonymous, empty session

    # --- the scorer's contract (ScorerTests, live) ---------------------------
    p_risky = churn_probability(tenure_months=3, monthly_charges=95,
                                support_tickets=4, contract="month-to-month")
    p_safe = churn_probability(tenure_months=3, monthly_charges=95,
                               support_tickets=4, contract="two-year")
    assert 0.0 <= p_risky <= 1.0, "probability must be a probability"
    assert p_risky > p_safe, "month-to-month must out-risk two-year"

    # --- the form flow (ViewTests, live) --------------------------------------
    assert suite.get("/score/").status_code == 200
    rows_before = Prediction.objects.count()
    assert suite.post("/score/", CUSTOMER).status_code == 200
    assert Prediction.objects.count() == rows_before + 1, "POST must log exactly one row"

    # --- the JSON API ---------------------------------------------------------
    ok = suite.post("/api/score/", data=json.dumps(CUSTOMER),
                    content_type="application/json")
    assert ok.status_code == 200
    assert 0.0 <= ok.json()["probability"] <= 1.0
    assert suite.post("/api/score/", data='{"monthly_charges": 95}',
                      content_type="application/json").status_code == 400

    # --- the auth boundary (HistoryTests, live) --------------------------------
    bounced = suite.get("/history/")
    assert bounced.status_code == 302
    assert bounced.headers["Location"] == "/accounts/login/?next=/history/"
    suite.force_login(analyst)
    member = suite.get("/history/")
    assert member.status_code == 200
    assert "month-to-month" in member.content.decode()

    print("✅ 9 assertions green — scorer, form, API, and the auth boundary")
    print("   (the same checks a CI run would make before letting this deploy)")
else:
    print("Django not installed — skipping.")

Bad Request: /api/score/


✅ 9 assertions green — scorer, form, API, and the auth boundary
   (the same checks a CI run would make before letting this deploy)


> 🧠 **Why the auth asserts matter most.** The pair *redirect-for-strangers / content-for-members* pins the security behaviour to the floor. A refactor that accidentally drops `@login_required` doesn't "probably get noticed" — it **fails CI** before it can leak a single row. Security you don't test is security you don't have.

---

### ✋ Quick exercise (~2 min) — Two more tests for the suite

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Two behaviours we demonstrated in §4 but never wrote down as tests: **(a)** `GET /api/score/` must return `405`, and **(b)** a payload with `tenure_months="three"` (with the other three fields valid) must return `400`. Write both asserts against `suite` — and, for each, say which *line* of `api_score` produces the status code.

In [21]:
# ✍️ Your turn 👇
# assert suite.get("/api/score/").status_code == ...

# garbled = suite.post("/api/score/", data=json.dumps({...}), content_type="application/json")
garbled = ...

# assert garbled.status_code == ...
# print("both failure modes are now pinned down ✅")

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    assert suite.get("/api/score/").status_code == 405

    garbled = suite.post("/api/score/",
                         data=json.dumps({"tenure_months": "three", "monthly_charges": 95,
                                          "support_tickets": 4, "contract": "month-to-month"}),
                         content_type="application/json")
    assert garbled.status_code == 400
    print("both failure modes are now pinned down ✅ —", garbled.json()["error"])
else:
    print("Django not installed — skipping.")
```

**(a)** comes from the `@require_POST` decorator — the view body never even runs. **(b)** comes from `int(payload["tenure_months"])` raising `ValueError`, caught by the `except (KeyError, ValueError, TypeError)` line and converted to a 400 with the reason in the body. Every status code in an API should be traceable to one specific line like this — if you can't point at the line, you don't know your own failure modes.
</details>

## 8. Production prep — the checklist and the box

Everything above runs on **dev settings**, and [deployment.md](deployment.md) is blunt about what must change before strangers can reach this app. The big three, plus the two serving habits:

| Setting | Dev (this lab) | Production |
|---|---|---|
| `DEBUG` | `True` | **`False`** — a `DEBUG=True` error page leaks settings, SQL and file paths to whoever caused the error |
| `SECRET_KEY` | a hard-coded lab string | **from an environment variable** — it signs sessions and password-reset tokens; leaked = every login forgeable |
| `ALLOWED_HOSTS` | `["testserver"]` | your real domain(s) — stops Host-header cache poisoning |
| server | (none — test client) | **gunicorn** behind Nginx — `runserver` is single-threaded, unhardened, dev-only |
| static files | (none) | `collectstatic` + WhiteNoise/Nginx |

> ⚠️ **Secrets live in the environment, never in code.** The ChurnScope settings file reads all three from env vars, so "going to production" is *setting* them, not editing code — and the secret never enters git, where it would outlive every attempt to delete it:
>
> ```bash
> export DJANGO_DEBUG=0
> export DJANGO_SECRET_KEY="$(python -c 'import secrets; print(secrets.token_urlsafe(50))')"
> export DJANGO_ALLOWED_HOSTS="churnscope.example.com"
> ```

Let's audit *this notebook's* settings against that checklist — and fail proudly, because red is correct for a lab:

In [22]:
if HAS_DJANGO:
    audit = [
        ("DEBUG is False", settings.DEBUG is False,
         "export DJANGO_DEBUG=0  — never leak tracebacks to strangers"),
        ("SECRET_KEY comes from the environment", "DJANGO_SECRET_KEY" in os.environ,
         "export DJANGO_SECRET_KEY=…  — generated, never committed"),
        ("ALLOWED_HOSTS names real domains",
         bool(settings.ALLOWED_HOSTS) and settings.ALLOWED_HOSTS not in (["*"], ["testserver"]),
         'export DJANGO_ALLOWED_HOSTS="churnscope.example.com"'),
        ("served by gunicorn, not runserver", False,
         "gunicorn churnscope.wsgi --bind 0.0.0.0:8000"),
        ("static files collected", False,
         "python manage.py collectstatic --noinput  (+ WhiteNoise or Nginx)"),
    ]
    for label, ok, fix in audit:
        print(f"{'✅' if ok else '❌'} {label}")
        if not ok:
            print(f"      fix: {fix}")
    print("\nAll red — and honestly so: these are dev settings doing dev work.")
    print("The example app reads the big three from env vars, so production is `export`, not editing.")
else:
    print("Django not installed — skipping.")

❌ DEBUG is False
      fix: export DJANGO_DEBUG=0  — never leak tracebacks to strangers
❌ SECRET_KEY comes from the environment
      fix: export DJANGO_SECRET_KEY=…  — generated, never committed
❌ ALLOWED_HOSTS names real domains
      fix: export DJANGO_ALLOWED_HOSTS="churnscope.example.com"
❌ served by gunicorn, not runserver
      fix: gunicorn churnscope.wsgi --bind 0.0.0.0:8000
❌ static files collected
      fix: python manage.py collectstatic --noinput  (+ WhiteNoise or Nginx)

All red — and honestly so: these are dev settings doing dev work.
The example app reads the big three from env vars, so production is `export`, not editing.


### The box — ChurnScope's real Dockerfile

The deployment chapter's final move is to **containerise** the app, and the example app ships a real, working [`Dockerfile`](example-app/Dockerfile). We load it the resilient way — local file first (you're inside the course repo), GitHub raw next (you're on Colab), inline copy last (you're on a plane) — and read it like a deployment checklist that happens to be executable:

In [23]:
# The exact contents of 17_django/example-app/Dockerfile, inlined so this
# section works with no repo checkout and no network. (An r-string, because
# the file contains trailing backslashes that must survive verbatim.)
DOCKERFILE_FALLBACK = r"""
# syntax=docker/dockerfile:1

# ---- Base image: small, official Python (same pattern as Module 14) ----
FROM python:3.12-slim

# Sensible Python defaults inside containers
ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1

# All later paths are relative to /app
WORKDIR /app

# Copy ONLY requirements first so the pip layer is cached until deps change
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Now copy the application code
COPY manage.py .
COPY churnscope ./churnscope
COPY scoring ./scoring

# Run as a non-root user (never run containers as root in production).
# appuser must own /app so SQLite can create db.sqlite3 there.
RUN useradd --create-home appuser && chown -R appuser /app
USER appuser

# Document the port the app listens on
EXPOSE 8000

# Container-level liveness check — stdlib only, since slim images have no curl.
# It hits the form page. With DJANGO_DEBUG=0 you must include 'localhost' in
# DJANGO_ALLOWED_HOSTS or the check 400s (see the run example in deployment.md).
HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \
  CMD python -c "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://localhost:8000/').status==200 else 1)"

# Migrate, then serve with gunicorn (the WSGI server; runserver is dev-only).
# Migrating at container start is fine for this single-container SQLite demo;
# in real life run migrations as a separate release step so several replicas
# don't race to migrate the same database. `exec` makes gunicorn PID 1 so it
# receives Docker's stop signals directly.
CMD ["sh", "-c", "python manage.py migrate --noinput && exec gunicorn churnscope.wsgi --bind 0.0.0.0:8000"]
"""[1:]


import urllib.request

LOCAL_DOCKERFILE = Path("example-app/Dockerfile")
RAW_URL = ("https://raw.githubusercontent.com/ChrisW09/Python-for-AI-Driven-Automation/"
           "main/17_django/example-app/Dockerfile")


def load_dockerfile() -> tuple[str, str]:
    """Local file → GitHub raw URL → inline constant. Offline never breaks."""
    if LOCAL_DOCKERFILE.exists():
        return LOCAL_DOCKERFILE.read_text(), f"local file: {LOCAL_DOCKERFILE}"
    try:
        with urllib.request.urlopen(RAW_URL, timeout=10) as resp:
            return resp.read().decode("utf-8"), "GitHub raw URL"
    except Exception:
        return DOCKERFILE_FALLBACK, "inline fallback constant"


dockerfile_text, loaded_from = load_dockerfile()
print(f"✅ Dockerfile loaded from {loaded_from}\n")
for no, line in enumerate(dockerfile_text.splitlines(), start=1):
    print(f"{no:>2} │ {line}")

✅ Dockerfile loaded from local file: example-app/Dockerfile

 1 │ # syntax=docker/dockerfile:1
 2 │ 
 3 │ # ---- Base image: small, official Python (same pattern as Module 12) ----
 4 │ FROM python:3.12-slim
 5 │ 
 6 │ # Sensible Python defaults inside containers
 7 │ ENV PYTHONUNBUFFERED=1 \
 8 │     PYTHONDONTWRITEBYTECODE=1
 9 │ 
10 │ # All later paths are relative to /app
11 │ WORKDIR /app
12 │ 
13 │ # Copy ONLY requirements first so the pip layer is cached until deps change
14 │ COPY requirements.txt .
15 │ RUN pip install --no-cache-dir -r requirements.txt
16 │ 
17 │ # Now copy the application code
18 │ COPY manage.py .
19 │ COPY churnscope ./churnscope
20 │ COPY scoring ./scoring
21 │ 
22 │ # Run as a non-root user (never run containers as root in production).
23 │ # appuser must own /app so SQLite can create db.sqlite3 there.
24 │ RUN useradd --create-home appuser && chown -R appuser /app
25 │ USER appuser
26 │ 
27 │ # Document the port the app listens on
28 │ EXPOSE 8000
29 │ 

Five decisions in that file are the whole craft of putting a model server in a box:

1. **`COPY requirements.txt` before `COPY <code>`.** Docker caches layer by layer; dependencies change rarely, code changes constantly — this ordering means a code-only change rebuilds in seconds because the slow `pip install` layer is reused. (The exact pattern from Module 14's backend image.)
2. **`USER appuser`** — the container does *not* run as root, so a compromised app can't own the box. Note the `chown`: SQLite needs to create its file in `/app`.
3. **`HEALTHCHECK` with stdlib urllib** — slim images have no `curl`; a Python one-liner asks the form page "are you alive?" every 30 s, and Docker (or Module 14's compose file) can restart the container when the answer stops being 200.
4. **`migrate && exec gunicorn`** at start — the container sets up its own schema, then `exec` makes gunicorn PID 1 so it receives stop signals directly. Fine for one container + SQLite; with replicas you run migrations as a separate release step so they don't race.
5. **What's *missing* is deliberate:** no secrets baked into the image (they arrive as `-e DJANGO_SECRET_KEY=…` at `docker run` time) and the SQLite file lives in the container's writable layer — ephemeral unless you bind-mount it. Real deployments swap in Postgres, which is exactly what Module 14's compose file does.

```bash
cd 17_django/example-app
docker build -t churnscope .
docker run -p 8000:8000 -e DJANGO_SECRET_KEY="change-me" \
    -e DJANGO_DEBUG=0 -e DJANGO_ALLOWED_HOSTS=localhost churnscope
```

> 🧠 **The hand-off.** With this image built, ChurnScope drops straight into Module 14's machinery: [`../14_cicd/lab01_docker_and_compose.ipynb`](../14_cicd/lab01_docker_and_compose.ipynb) puts it behind Nginx with Postgres, [`../14_cicd/lab02_ci_pipeline_github_actions.ipynb`](../14_cicd/lab02_ci_pipeline_github_actions.ipynb) builds and pushes it on every commit, and [`../14_cicd/lab03_deploy_dns_https_monitoring.ipynb`](../14_cicd/lab03_deploy_dns_https_monitoring.ipynb) gives it a domain and a certificate. Same pipeline you already know — a Django image instead of a FastAPI one.

## 🧪 Practice exercises

Everything below builds on objects already in memory: `pipeline`, `_model`, `churn_probability`, `MODEL_VERSION`, `client`, `analyst`, `urlconf`, `TEMPLATE_REGISTRY`…

### Exercise 1 — ⭐ Read the model's mind

The artifact is a pipeline, and pipelines can explain themselves. Pull the fitted `LogisticRegression` out of `pipeline.named_steps` and pair its `coef_[0]` with `pipeline.named_steps["prep"].get_feature_names_out()` in a sorted `pd.Series`. Which feature pushes churn risk up hardest? Which pulls it down? Compare with the coefficients of `baseline_churn_probability` — did the model recover the story it was never told?

In [24]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
prep = pipeline.named_steps["prep"]
clf = pipeline.named_steps["clf"]

weights = pd.Series(clf.coef_[0], index=prep.get_feature_names_out()).sort_values()
print(weights.round(3))
print(f"\nrisk-raising champion : {weights.idxmax()}")
print(f"risk-lowering champion: {weights.idxmin()}")
```

The one-hot column for `month-to-month` carries the biggest positive weight and tenure (with `two-year`) the biggest negative ones — the ground-truth process (`+1.4·contract_risk`, `−0.05·tenure`) shining through the noise. Mind one reading trap: the numeric coefficients apply to **standardised** features (the `StandardScaler` runs first), so "0.45 per standard deviation of monthly charges", not "per euro". Model explanation always happens *after* preprocessing — one more reason the preprocessing must live inside the artifact.
</details>

### Exercise 2 — ⭐⭐ A `/api/model/` metadata endpoint

"Which model is this server running?" should never require SSH. Add `GET /api/model/` returning JSON with: `model_version`, the artifact filename, its size in KB, its `trained_at` timestamp (`MODEL_PATH.stat().st_mtime` → ISO string), the `FEATURES` list, and the estimator class name. Wire it into `urlconf.urlpatterns` (then `clear_url_caches()`) and verify with the client. This endpoint is the first thing you'll be glad exists when a score looks wrong next quarter.

In [25]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    import datetime as dt

    def api_model(request):
        stat = MODEL_PATH.stat()
        return JsonResponse({
            "model_version": MODEL_VERSION,
            "artifact": MODEL_PATH.name,
            "size_kb": round(stat.st_size / 1024, 1),
            "trained_at": dt.datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
            "features": FEATURES,
            "estimator": type(pipeline.named_steps["clf"]).__name__,
        })

    urlconf.urlpatterns = [u for u in urlconf.urlpatterns if str(u.pattern) != "api/model/"]
    urlconf.urlpatterns.append(path("api/model/", api_model, name="api_model"))
    clear_url_caches()

    print(client.get("/api/model/").json())
```

A `GET` with no side effects needs neither `@csrf_exempt` nor `@require_POST`. The filter-then-append dance keeps the cell idempotent (Lab 1's routing drill). In real systems this grows into a `/health` + `/info` pair that load balancers and dashboards poll — and the `trained_at` field is your early-warning light for "the retraining job has silently stopped running".
</details>

### Exercise 3 — ⭐⭐ Logout is a POST

[auth-and-history.md](auth-and-history.md) ends on a modern gotcha: Django 5 **removed GET logout** — destroying a session is a state change, so it must be a POST (otherwise a prefetching browser or a malicious `<img>` tag could log users out). Prove all three claims with the client: log a fresh client in with `force_login`, show `GET /accounts/logout/` returns **405**, show `POST /accounts/logout/` returns **302** to `LOGOUT_REDIRECT_URL` (`/score/`), and show `/history/` bounces to the login page again afterwards.

In [26]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    member = Client()
    member.force_login(analyst)
    assert member.get("/history/").status_code == 200      # in

    get_try = member.get("/accounts/logout/")
    print("GET  /accounts/logout/ →", get_try.status_code, "(Django 5: not allowed)")
    assert get_try.status_code == 405

    post_try = member.post("/accounts/logout/")
    print("POST /accounts/logout/ →", post_try.status_code, "→", post_try.headers["Location"])
    assert post_try.status_code == 302
    assert post_try.headers["Location"] == "/score/"       # LOGOUT_REDIRECT_URL

    assert member.get("/history/").status_code == 302      # out again
    print("session destroyed — /history/ bounces to login again ✅")
```

The same rule you met with forms: **reads are GET, state changes are POST.** That's why the example app's base template renders logout as a one-button `<form method="post">{% csrf_token %}…</form>` instead of a link — and why the CSRF token guards logout too: your session is state worth protecting in *both* directions.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

A colleague read the chapter's serving snippet — which builds a plain list of features, because *that* model was trained on a bare array — and "simplified" our scorer accordingly. The cell **crashes**. Run it, read the error from the bottom up, explain what contract was broken, and write the fix. (Hint: how was *our* pipeline trained — on what kind of object, with what column names?)

In [27]:
# ⚠️ THIS CELL INTENTIONALLY ERRORS — that's the exercise. Read the traceback!
MODEL = _model      # the loaded pipeline from §3


def quick_probability(tenure_months, monthly_charges, support_tickets, contract):
    """Score one customer 'the quick way' — what could possibly go wrong?"""
    row = [[CONTRACT_RISK.get(contract, 0.0), tenure_months,
            monthly_charges, support_tickets]]
    return float(MODEL.predict_proba(row)[0, 1])


print(quick_probability(3, 95, 4, "month-to-month"))

ValueError: Specifying the columns using strings is only supported for dataframes.

<details>
<summary>💡 <b>Solution</b></summary>

**The broken contract: the serving input must match the training schema.** Our pipeline starts with a `ColumnTransformer` that selects columns **by name** (`["contract"]`, the numeric list) — that only works on a DataFrame, so a bare list of lists raises `ValueError: Specifying the columns using strings is only supported for dataframes`. And there's a second, sneakier bug hiding behind the first: the colleague *pre-encoded* the contract as a risk number, but our artifact does its own one-hot encoding inside — even if the list were accepted, the features would be in the wrong shape and meaning.

```python
def quick_probability_fixed(tenure_months, monthly_charges, support_tickets, contract):
    row = pd.DataFrame([{
        "tenure_months": tenure_months,
        "monthly_charges": monthly_charges,
        "support_tickets": support_tickets,
        "contract": contract,                 # raw value — the pipeline encodes it
    }])
    return float(MODEL.predict_proba(row)[0, 1])

print(f"fixed: {quick_probability_fixed(3, 95, 4, 'month-to-month'):.4f}")
```

Which is character-for-character what §3's `churn_probability` does — same columns, same names, raw values, one row. The chapter's list-based snippet is correct *for its model* (trained on a bare array with hand-built features); ours ships preprocessing inside the artifact precisely so serving code never re-implements it. **Whatever schema fitted the pipeline is the schema the endpoint must rebuild — the artifact remembers.**
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ Version every row

[serving-a-model.md](serving-a-model.md)'s parting advice: add `model_version` to `Prediction`, so every score is forever traceable to the artifact that made it. In the real project that's a field + `makemigrations` + `migrate`. In a live notebook you can perform the same schema evolution by hand — three moves:

1. `Prediction.add_to_class("model_version", models.CharField(max_length=40, default=""))` — attach the field to the *class* (what editing `models.py` does),
2. `connection.schema_editor().add_field(...)` — issue the `ALTER TABLE` (what the migration would run),
3. backfill existing rows with one `update()`, then confirm new rows carry the version (adjust one of the views to pass `model_version=MODEL_VERSION`).

Finish with a per-version count: `values("model_version").annotate(n=Count("id"))`.

In [28]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    from django.db.models import Count

    # 1) the class learns the field (≙ editing models.py)
    if not any(f.name == "model_version" for f in Prediction._meta.fields):
        Prediction.add_to_class(
            "model_version", models.CharField(max_length=40, default=""))
        # 2) the table learns the column (≙ the migration's ALTER TABLE)
        with connection.schema_editor() as editor:
            editor.add_field(Prediction, Prediction._meta.get_field("model_version"))

    # 3) history gets a value too — one UPDATE, no loop
    backfilled = Prediction.objects.filter(model_version="").update(
        model_version=MODEL_VERSION)
    print(f"backfilled {backfilled} rows")

    # new rows carry it automatically once views pass it:
    Prediction.objects.create(model_version=MODEL_VERSION, probability=0.42,
                              will_churn=False, tenure_months=12,
                              monthly_charges=70, support_tickets=1,
                              contract="one-year")
    print(list(Prediction.objects.values("model_version").annotate(n=Count("id"))))
```

The `if not any(...)` guard makes the cell idempotent — `add_to_class` twice would try to `ALTER TABLE` twice. A backfill marked with the *current* version is a small lie (those rows may predate it); real teams backfill with `"unversioned"` or the best-known version, and the honest lesson stands: **add the version column before the first deployment, not after the first incident.** From here, updating `api_score` and `index` to pass `model_version=MODEL_VERSION` is a two-line edit each.
</details>

### Stretch exercise B — ⭐⭐⭐ APIs don't redirect — a JSON-aware login guard

Put `@login_required` on a JSON endpoint and an unauthenticated `curl` receives… a **302 to an HTML login page** — useless to a script, and it can hide real failures (many HTTP clients follow the redirect and report a confusing 200). Write a decorator `api_login_required(view)` that returns `JsonResponse({"error": "authentication required"}, status=401)` for anonymous callers and otherwise calls through (`functools.wraps` keeps the view's name). Use it to protect a new `GET /api/stats/` endpoint — total predictions plus per-contract `n`, `avg_probability`, and `churn_share` (hint: `Avg("will_churn", output_field=FloatField())` works because booleans store as 0/1). Verify: anonymous → `401`, `force_login` → `200`.

In [29]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    import functools

    from django.db.models import Avg, Count, FloatField

    def api_login_required(view):
        @functools.wraps(view)
        def wrapper(request, *args, **kwargs):
            if not request.user.is_authenticated:
                return JsonResponse({"error": "authentication required"}, status=401)
            return view(request, *args, **kwargs)
        return wrapper

    @api_login_required
    def api_stats(request):
        per_contract = list(
            Prediction.objects.values("contract")
            .annotate(n=Count("id"),
                      avg_probability=Avg("probability"),
                      churn_share=Avg("will_churn", output_field=FloatField()))
            .order_by("-avg_probability"))
        return JsonResponse({"total": Prediction.objects.count(),
                             "per_contract": per_contract})

    urlconf.urlpatterns = [u for u in urlconf.urlpatterns if str(u.pattern) != "api/stats/"]
    urlconf.urlpatterns.append(path("api/stats/", api_stats, name="api_stats"))
    clear_url_caches()

    anon = Client()
    print("anonymous →", anon.get("/api/stats/").status_code)
    anon.force_login(analyst)
    print(json.dumps(anon.get("/api/stats/").json(), indent=2))
```

Same *policy* as `@login_required`, different *dialect*: humans get redirected to a form, machines get a status code they can branch on (401 = "authenticate", not 403 = "you may never"). This dialect split is exactly why DRF and Django Ninja exist — their authentication classes speak JSON natively, with token/session/OAuth flavours. The stats queryset itself is the ORM's GROUP-BY pattern from Lab 1, now wearing a production shirt.
</details>

### Stretch exercise C — ⭐⭐⭐ Champion vs challenger

You now hold *two* scorers with identical signatures: `churn_probability` (the trained pipeline — the **champion**) and `baseline_churn_probability` (the hand-tuned stand-in — a **challenger**). Build `POST /api/score/v2/` that reads an optional `?model=` query parameter (`logreg` default, `baseline` the alternative), routes to the matching scorer from a `SCORERS` dict, rejects unknown names with a `400` that lists the valid choices, and includes `"model": <name>` in the response. Then score the same customer under both and note the disagreement — that gap is what real champion/challenger (A/B) setups measure at scale.

In [30]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
if HAS_DJANGO:
    SCORERS = {"logreg": churn_probability, "baseline": baseline_churn_probability}

    @csrf_exempt
    @require_POST
    def api_score_v2(request):
        name = request.GET.get("model", "logreg")
        scorer = SCORERS.get(name)
        if scorer is None:
            return JsonResponse({"error": f"unknown model {name!r}",
                                 "choices": list(SCORERS)}, status=400)
        try:
            payload = json.loads(request.body or "{}")
            data = {"tenure_months": int(payload["tenure_months"]),
                    "monthly_charges": float(payload["monthly_charges"]),
                    "support_tickets": int(payload["support_tickets"]),
                    "contract": str(payload["contract"])}
        except (KeyError, ValueError, TypeError) as exc:
            return JsonResponse({"error": f"invalid request: {exc}"}, status=400)
        p = scorer(**data)
        return JsonResponse({"model": name, "probability": round(p, 4),
                             "will_churn": p >= THRESHOLD})

    urlconf.urlpatterns = [u for u in urlconf.urlpatterns if str(u.pattern) != "api/score/v2/"]
    urlconf.urlpatterns.append(path("api/score/v2/", api_score_v2, name="api_score_v2"))
    clear_url_caches()

    for name in ("logreg", "baseline", "gpt-9000"):
        r = client.post(f"/api/score/v2/?model={name}", data=json.dumps(CUSTOMER),
                        content_type="application/json")
        print(f"{name:<9} → {r.status_code} {r.json()}")
```

Because both scorers honour the same **seam** (same keyword-only signature, same 0–1 return), swapping models is a dict lookup — the strongest argument this lab makes for designing the seam first. Production versions of this pattern route a *percentage of traffic* rather than a query param, log which model served each request (Stretch A's column suddenly earns its keep), and compare *outcomes*, not just probabilities.
</details>

## 🎁 Bonus mini-project — Batch CSV scoring

The retention team doesn't score customers one by one — they show up with a CSV of five hundred. Build a `/batch/` page:

- a `BatchForm` with a single `forms.FileField`,
- a view that reads the upload with `pd.read_csv`, scores **all rows in one vectorised `predict_proba` call** (never a Python loop over rows — the artifact is happiest with a whole DataFrame),
- logs everything with **one** `Prediction.objects.bulk_create(...)` (one multi-row INSERT, not N round-trips — the ORM cousin of vectorisation),
- renders a summary template: how many scored, how many likely churners, average probability, and the top-3 riskiest rows.

Test it notebook-style with `django.core.files.uploadedfile.SimpleUploadedFile` wrapping `customers[FEATURES].head(20).to_csv(index=False).encode()`. Stretch the stretch: reject files missing a required column with a form error instead of a 500.

In [31]:
# Your code here  👇

<details>
<summary>💡 <b>Solution sketch</b></summary>

```python
if HAS_DJANGO:
    from django.core.files.uploadedfile import SimpleUploadedFile

    class BatchForm(forms.Form):
        file = forms.FileField(help_text="CSV with the four feature columns")

    TEMPLATE_REGISTRY["scoring/batch.html"] = """\
<h1>Batch scoring</h1>
<form method="post" enctype="multipart/form-data">
  {% csrf_token %}
  {{ form.as_p }}
  <button type="submit">Score the file</button>
</form>
{% if summary %}<p id="summary">{{ summary.n }} customers scored — {{ summary.churners }} likely churners
(avg p = {{ summary.avg_p|floatformat:3 }}).</p>
<ol>{% for row in summary.riskiest %}<li>{{ row.contract }} — {{ row.probability|floatformat:2 }}</li>{% endfor %}</ol>
{% endif %}"""

    def batch_score(request):
        summary, form = None, BatchForm()
        if request.method == "POST":
            form = BatchForm(request.POST, request.FILES)
            if form.is_valid():
                frame = pd.read_csv(form.cleaned_data["file"])
                probs = _model.predict_proba(frame[FEATURES])[:, 1]   # ONE vectorised call
                frame["probability"] = probs.round(4)
                frame["will_churn"] = probs >= THRESHOLD
                Prediction.objects.bulk_create(                       # ONE multi-row INSERT
                    Prediction(**row) for row in frame.to_dict("records"))
                top = frame.sort_values("probability", ascending=False).head(3)
                summary = {"n": len(frame), "churners": int(frame["will_churn"].sum()),
                           "avg_p": float(frame["probability"].mean()),
                           "riskiest": top.to_dict("records")}
                form = BatchForm()
        return render(request, "scoring/batch.html", {"form": form, "summary": summary})

    urlconf.urlpatterns = [u for u in urlconf.urlpatterns if str(u.pattern) != "batch/"]
    urlconf.urlpatterns.append(path("batch/", batch_score, name="batch_score"))
    clear_url_caches()

    csv_bytes = customers[FEATURES].head(20).to_csv(index=False).encode()
    upload = SimpleUploadedFile("customers.csv", csv_bytes, content_type="text/csv")
    before = Prediction.objects.count()
    resp = client.post("/batch/", {"file": upload})
    print("POST /batch/ →", resp.status_code, f"· rows {before} → {Prediction.objects.count()}")
    for line in resp.content.decode().splitlines():
        if "summary" in line or "<li>" in line:
            print("   ", line)
```

Twenty rows, two database statements, one `predict_proba`. For the column check, validate inside the form (a `clean_file` method that reads the header and raises `forms.ValidationError` listing what's missing) so the user sees a form error, not a traceback. And if "five hundred rows" becomes "five million", remember the chapter's rule — that's no longer request-path work; it's a background job (Module 13's schedulers) that writes results the page merely *reads*.
</details>

## 🧠 Key takeaways

> 🧭 **The story in one line.** We turned a trained scikit-learn pipeline into a product: **train offline → ship a versioned file → load once at startup → predict per request → log every prediction** — served to robots as JSON and to humans as a form, with the ledger locked behind a login and a Dockerfile waiting at the exit.

1. **The model is a file plus a seam.** `joblib.dump` ships preprocessing and estimator as one artifact; a keyword-only function with a stable signature (`churn_probability(...)`) hides all ML from all web. Swapping models never touches views.
2. **Load once, never per request — and *never* train in a view.** The real hook is `ScoringConfig.ready()`; the notebook equivalent is one cell run once. Slow, heavy, retryable work lives off the request path entirely.
3. **Version everything that predicts.** A `MODEL_VERSION` next to the artifact, logged with every row, is the difference between an audit trail and an archaeology dig.
4. **Serving-time schema = training-time schema.** The pipeline was fitted on a named-column DataFrame, so the endpoint rebuilds exactly that — and `handle_unknown="ignore"` decides *at training time* how unseen categories fail at 3 a.m.
5. **An API is defined by its failures**: 400 with a reason for bad input, 405 for wrong methods — each status traceable to one line of the view.
6. **`@csrf_exempt` is a scalpel.** CSRF defends cookie-carrying browsers; a stateless JSON API steps outside that shield and must adopt token auth instead. Form views keep `{% csrf_token %}`, always.
7. **Auth is assembled, not written**: sessions + two middleware + three settings + one `include` + one decorator. `request.user` *is* the login state; `create_user` hashes; `force_login` is the test-suite fast path; authentication ≠ authorisation.
8. **The test client compresses the app into milliseconds** — status codes, probability bounds, and the redirect-for-strangers/content-for-members pair that makes dropped decorators fail CI instead of leaking data.
9. **Production is a checklist, then a box**: `DEBUG=False`, `SECRET_KEY` and hosts from env, gunicorn + collectstatic — then a Dockerfile with layer-cached deps, a non-root user, a healthcheck, and no secrets inside.

## ✅ Self-assessment

- [ ] I can train a sklearn pipeline and explain why preprocessing must travel *inside* the artifact
- [ ] I can say where a real Django app loads its model, and why per-request loading (or training) is the classic mistake
- [ ] I can write a JSON endpoint that validates, predicts, logs, and fails with clean 400/405s
- [ ] I can explain what `@csrf_exempt` disables, why the API needed it, and what production uses instead
- [ ] I can protect a view with `@login_required` and trace the 302 → login → 200 arc with the test client
- [ ] I can explain what `create_user` stores in the password column and why `force_login` is fine in tests
- [ ] I can audit settings for production and defend each of the Dockerfile's five decisions

## 🚀 Next step

Three roads, in order:

1. **Run the real thing.** [`example-app/`](example-app/) is this notebook as files: the [module README](README.md)'s 2-minute instructions (`migrate` → `runserver`) put ChurnScope in your browser — score a customer, hit `/history/`, get bounced, `createsuperuser`, sign in, and see your ledger. Then perform this lab's §3 on it for real: [exercises.md](exercises.md) Exercise 5 is exactly that swap.
2. **Read [deployment.md](deployment.md) beside the Dockerfile** and actually `docker build` the image — §8's five decisions are better felt than read.
3. **Ship it through Module 14**: [`../14_cicd/lab01_docker_and_compose.ipynb`](../14_cicd/lab01_docker_and_compose.ipynb) → [`../14_cicd/lab02_ci_pipeline_github_actions.ipynb`](../14_cicd/lab02_ci_pipeline_github_actions.ipynb) → [`../14_cicd/lab03_deploy_dns_https_monitoring.ipynb`](../14_cicd/lab03_deploy_dns_https_monitoring.ipynb) — the same conveyor belt, now carrying *your* model in *your* app.

> 🚀 A model in a notebook is a finding. A model behind a login, with a ledger and a version string, is a *product*. You just built the second kind.